# SARIMA / AutoARIMA Training

Theory-focused SARIMA experiment notebook for the Walmart sales forecasting project.

What it does:
- uses `statsforecast` AutoARIMA with weekly seasonality (`season_length=52`);
- trains only on a small high-volume subset because per-series classical models do not scale well to roughly 3,000 Walmart series;
- evaluates the subset with WMAE over the same 39-week horizon;
- logs MLflow stages and creates a concise theory section for the README.

This is intentionally not the main winning model. Its value is interpretability and report coverage.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dataloader import load_raw
from metrics import wmae
from validation import expanding_window_folds

warnings.filterwarnings("ignore")
DATA_DIR = ROOT / "data"
EXPERIMENT_NAME = "SARIMA_Training"
MODEL_ALIAS = "AutoARIMA"
HORIZON = 39
SEASON_LENGTH = 52
N_SERIES = 8

In [ ]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=True)
except ImportError:
    pass

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

# Some local environments set a dead proxy (for example 127.0.0.1:9),
# which prevents MLflow from reaching DagsHub. Clear it before tracking calls.
for key in [
    "HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
    "ALL_PROXY", "all_proxy",
]:
    os.environ.pop(key, None)

mlflow.set_experiment(EXPERIMENT_NAME)


# Do not let temporary DagsHub/DNS failures crash long training cells.
# Failed log calls are printed; rerun the cell later on a stable connection if a required metric is missing remotely.
def _safe_mlflow_call(fn, label):
    def wrapped(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            print(f"[MLflow warning] {label} failed: {type(exc).__name__}: {str(exc)[:300]}")
            return None
    return wrapped

mlflow.log_param = _safe_mlflow_call(mlflow.log_param, "log_param")
mlflow.log_params = _safe_mlflow_call(mlflow.log_params, "log_params")
mlflow.log_metric = _safe_mlflow_call(mlflow.log_metric, "log_metric")
mlflow.log_metrics = _safe_mlflow_call(mlflow.log_metrics, "log_metrics")
mlflow.log_artifact = _safe_mlflow_call(mlflow.log_artifact, "log_artifact")
mlflow.log_artifacts = _safe_mlflow_call(mlflow.log_artifacts, "log_artifacts")

# Run this cell once before the stage cells when you want MLflow to group
# the required Cleaning -> Feature Selection -> CV -> HPO -> Final runs.
parent_run = mlflow.start_run(run_name="SARIMA_Workflow")

## Theory Notes for README

SARIMA extends ARIMA with seasonal terms. ARIMA combines autoregression, differencing, and moving-average residual correction. SARIMA adds seasonal versions of those terms, which matters for weekly Walmart data because sales have a strong 52-week yearly cycle.

ACF and PACF are diagnostic plots. ACF shows correlation with previous lags; PACF tries to isolate the direct correlation at a lag after accounting for shorter lags. Differencing is used when the mean level or seasonal level is not stable.

The limitation is scale: a separate SARIMA model per Store-Dept series means thousands of independent fits, weak sharing across sparse series, and poor use of tabular exogenous features like store type, markdowns, and CPI. That is why boosted trees and global neural models are the practical main models.

In [ ]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)
train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

def add_unique_id(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["unique_id"] = out["Store"].astype(str) + "_" + out["Dept"].astype(str)
    out["ds"] = pd.to_datetime(out["Date"])
    return out

train_uid = add_unique_id(train_raw)
top_ids = (
    train_uid.groupby("unique_id")["Weekly_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(N_SERIES)
    .index
    .tolist()
)
subset_raw = train_uid[train_uid["unique_id"].isin(top_ids)].copy()

def to_statsforecast(frame: pd.DataFrame) -> pd.DataFrame:
    frame = add_unique_id(frame) if "unique_id" not in frame.columns else frame.copy()
    return frame[["unique_id", "ds", "Weekly_Sales"]].rename(columns={"Weekly_Sales": "y"})

## MLflow Stage: Cleaning

The subset uses the highest-volume Store-Dept series so the classical model has enough signal and the notebook stays fast.

In [ ]:
with mlflow.start_run(run_name="SARIMA_Cleaning", nested=True):
    lengths = subset_raw.groupby("unique_id").size()
    mlflow.log_param("subset_rule", f"top_{N_SERIES}_series_by_total_sales")
    mlflow.log_param("negative_sales_training_policy", "keep")
    mlflow.log_metric("subset_series_count", int(len(top_ids)))
    mlflow.log_metric("min_subset_series_length", int(lengths.min()))
    pd.Series(top_ids, name="unique_id").to_csv(ROOT / "models" / "sarima_subset_series.csv", index=False)
    mlflow.log_artifact(str(ROOT / "models" / "sarima_subset_series.csv"))

## MLflow Stage: Feature Selection

Classical SARIMA uses only the target history. Exogenous variables are intentionally excluded in this theory-focused baseline.

In [ ]:
with mlflow.start_run(run_name="SARIMA_Feature_Selection", nested=True):
    mlflow.log_param("uses_exogenous_features", False)
    mlflow.log_param("season_length", SEASON_LENGTH)
    mlflow.log_param("model_family", "per_series_classical_time_series")

In [ ]:
def fit_predict_autoarima(train_frame: pd.DataFrame, horizon: int = HORIZON) -> pd.DataFrame:
    sf_train = to_statsforecast(train_frame)
    model = StatsForecast(
        models=[AutoARIMA(season_length=SEASON_LENGTH, alias=MODEL_ALIAS)],
        freq="W-FRI",
        n_jobs=-1,
    )
    model.fit(sf_train)
    return model.predict(h=horizon)

def cv_sarima_subset() -> list[float]:
    scores = []
    for fold, (tr_mask, val_mask) in enumerate(expanding_window_folds(train_raw["Date"], n_splits=3, horizon=HORIZON), start=1):
        fold_train = train_raw.loc[tr_mask].copy()
        fold_val = train_raw.loc[val_mask].copy()
        fold_train = add_unique_id(fold_train)
        fold_val = add_unique_id(fold_val)
        fold_train = fold_train[fold_train["unique_id"].isin(top_ids)].copy()
        fold_val = fold_val[fold_val["unique_id"].isin(top_ids)].copy()
        preds = fit_predict_autoarima(fold_train, horizon=HORIZON)
        scored = fold_val[["unique_id", "ds", "Weekly_Sales", "IsHoliday"]].merge(preds, on=["unique_id", "ds"], how="left")
        pred = np.clip(scored[MODEL_ALIAS].to_numpy(), 0, None)
        scores.append(wmae(scored["Weekly_Sales"], pred, scored["IsHoliday"]))
    return scores

## MLflow Stage: Cross-Validation

In [ ]:
with mlflow.start_run(run_name="SARIMA_CrossValidation", nested=True):
    cv_scores = cv_sarima_subset()
    for i, score in enumerate(cv_scores, start=1):
        mlflow.log_metric(f"fold_{i}_subset_wmae", score)
    mlflow.log_metric("mean_subset_cv_wmae", float(np.mean(cv_scores)))

## MLflow Stage: HPO

AutoARIMA performs its own order search, so there is no large external HPO loop. The only deliberate choice here is weekly seasonality (`m=52`).

In [ ]:
with mlflow.start_run(run_name="SARIMA_HPO", nested=True):
    mlflow.log_param("hpo_strategy", "AutoARIMA_internal_order_search")
    mlflow.log_param("season_length", SEASON_LENGTH)
    mlflow.log_param("external_trials", 0)

## MLflow Stage: Final Subset Model

This final stage trains the subset model and stores subset forecasts as an artifact. It is not meant to be registered as the production model.

In [ ]:
with mlflow.start_run(run_name="SARIMA_Final", nested=True):
    final_preds = fit_predict_autoarima(subset_raw, horizon=HORIZON)
    final_preds.to_csv(ROOT / "models" / "sarima_subset_forecast.csv", index=False)
    mlflow.log_param("production_candidate", False)
    mlflow.log_artifact(str(ROOT / "models" / "sarima_subset_forecast.csv"))

mlflow.end_run()

## README Takeaways

- SARIMA is interpretable and appropriate for explaining trend, differencing, residual autocorrelation, and yearly seasonality.
- It is not practical as the main solution because Walmart has thousands of Store-Dept series and SARIMA fits them separately.
- It also does not naturally exploit shared tabular predictors like store type, markdowns, CPI, unemployment, and holiday-specific effects.